# Notebook ٧: Text Classification من الصفر

Logistic Regression بإيدينا بالـ NumPy ثم بـ scikit-learn، على مقالات رياضية حقيقية (20 Newsgroups). كل خلية اتشغلت فعلًا.

## ١. الداتا

مقالات من 20 Newsgroups: بيسبول (0) وهوكي (1). بنشيل الـ headers عشان الموديل ما يغشش من اسم الجروب.

In [ ]:
# المصفوفات
import numpy as np
# داتا نصوص جاهزة
from sklearn.datasets import fetch_20newsgroups
# الفئتين
cats = ["rec.sport.baseball", "rec.sport.hockey"]
# نشيل أي ميتاداتا
kw = dict(categories=cats, remove=("headers", "footers", "quotes"))
# داتا التدريب
tr = fetch_20newsgroups(subset="train", **kw)
# داتا الاختبار
te = fetch_20newsgroups(subset="test", **kw)
# الأحجام وأسماء الفئات
print(len(tr.data), len(te.data), tr.target_names)
# عدد مقالات كل فئة
print(np.bincount(tr.target))
# بداية مقال للتجربة
print(tr.data[3][:120].replace("\n", " "))


## ٢. تحويل النص لمتجهات

بنعد 2000 كلمة هي الأكتر تكرارًا (binary: الكلمة موجودة ولا لأ).

In [ ]:
# معدّ الكلمات
from sklearn.feature_extraction.text import CountVectorizer
# 2000 كلمة من غير stop words
cv = CountVectorizer(max_features=2000, binary=True, stop_words="english")
# تدريب: مصفوفة عادية
Xtr = cv.fit_transform(tr.data).toarray().astype("float64")
# اختبار بنفس المفردات
Xte = cv.transform(te.data).toarray().astype("float64")
# الـ labels كأرقام
ytr, yte = tr.target.astype("float64"), te.target.astype("float64")
# الأشكال
print(Xtr.shape, Xte.shape)


## ٣. Logistic Regression بإيدينا

P(y=1) = sigmoid(w·x + b)، ونقلل الـ cross-entropy loss بالـ gradient descent (من فصل SLP 4).

In [ ]:
# دالة sigmoid
def sigmoid(z):
# بتحول أي رقم لاحتمال بين 0 و1
    return 1 / (1 + np.exp(-z))
# cross-entropy
def loss_fn(p, y):
# رقم صغير يمنع log(0)
    eps = 1e-9
# متوسط الخسارة
    return -np.mean(y * np.log(p + eps) + (1 - y) * np.log(1 - p + eps))
# نبدأ بأوزان صفر
w, b = np.zeros(Xtr.shape[1]), 0.0
# معدل التعلم
lr = 0.5
# 301 خطوة تدريب
for step in range(301):
# احتمالات التدريب
    p = sigmoid(Xtr @ w + b)
# gradient الأوزان
    gw = Xtr.T @ (p - ytr) / len(ytr)
# gradient الـ bias
    gb = np.mean(p - ytr)
# نحدّث الأوزان عكس الـ gradient
    w -= lr * gw
# نحدّث الـ bias
    b -= lr * gb
# كل 100 خطوة
    if step % 100 == 0:
# دقة التدريب
        acc = np.mean((p > 0.5) == ytr)
# نطبع الخسارة والدقة
        print(f"step {step:3}: loss={loss_fn(p, ytr):.4f} train_acc={acc:.3f}")
# احتمالات الاختبار
pte = sigmoid(Xte @ w + b)
# دقة الاختبار
print("test accuracy:", round(float(np.mean((pte > 0.5) == yte)), 3))


## ٤. نفس الشيء بـ scikit-learn

بنتأكد إن اللي كتبناه بإيدينا ماشي صح بمقارنته بالمكتبة.

In [ ]:
# الموديل الجاهز
from sklearn.linear_model import LogisticRegression
# تدريب
sk = LogisticRegression(max_iter=1000).fit(Xtr, ytr)
# دقة الاختبار
print("sklearn test accuracy:", round(sk.score(Xte, yte), 3))


## ٥. Precision وRecall وF1 بإيدينا

الدقة (accuracy) لوحدها ممكن تخدع، فنحسب المقاييس من الـ confusion matrix.

In [ ]:
# توقعات الموديل
pred = (pte > 0.5).astype(int)
# الـ labels الحقيقية
yt = yte.astype(int)
# توقع هوكي وهو هوكي
tp = int(((pred == 1) & (yt == 1)).sum())
# توقع هوكي وهو بيسبول
fp = int(((pred == 1) & (yt == 0)).sum())
# توقع بيسبول وهو هوكي
fn = int(((pred == 0) & (yt == 1)).sum())
# توقع بيسبول وهو بيسبول
tn = int(((pred == 0) & (yt == 0)).sum())
# precision وrecall
prec, rec = tp / (tp + fp), tp / (tp + fn)
# F1: المتوسط التوافقي
f1 = 2 * prec * rec / (prec + rec)
# confusion matrix
print([[tn, fp], [fn, tp]])
# القيم التلاتة
print(round(prec, 3), round(rec, 3), round(f1, 3))


In [ ]:
# نفس المقاييس من sklearn
from sklearn.metrics import precision_score, recall_score, f1_score
# للمقارنة
print(round(precision_score(yt, pred), 3), round(recall_score(yt, pred), 3), round(f1_score(yt, pred), 3))


## ٦. تفسير الموديل

الأوزان بتقول أنهي كلمات بتدفع ناحية كل فئة.

In [ ]:
# الكلمات
vocab = np.array(cv.get_feature_names_out())
# ترتيب الأوزان من الأقل للأعلى
order = np.argsort(w)
# كلمات بتدفع لبيسبول
print("baseball:", list(vocab[order[:8]]))
# كلمات بتدفع لهوكي
print("hockey:  ", list(vocab[order[-8:]]))


## ٧. Regularization

L2 بيمنع الأوزان تكبر زيادة، وبيقلل الـ overfitting. C الصغير = تنظيم أقوى.

In [ ]:
# قيم C مختلفة
for C in [0.001, 0.01, 0.1, 1, 10]:
# تدريب بالقيمة دي
    m = LogisticRegression(C=C, max_iter=1000).fit(Xtr, ytr)
# دقة التدريب والاختبار
    print(f"C={C:<6} train={m.score(Xtr, ytr):.3f} test={m.score(Xte, yte):.3f}")
